In [ ]:
import pandas as pd
import json

In [ ]:
df=pd.read_parquet('Dataset_Master_Predictions_2026.parquet')
df.head()

In [ ]:
df.shape

#### Vérification des SIREN pour sample

In [ ]:
# 1. Préparation des données (Assurer le format String pour éviter les pertes de 0)
df['SIREN'] = df['SIREN'].astype(str).str.strip().str.zfill(9)
col_dep = "Code du département de l'établissement"

# 2. Filtrage des SIREN commençant par '0'
df_zero = df[df['SIREN'].str.startswith('0')].copy()
nb_zero = len(df_zero)

print(f"🔍 ANALYSE DES {nb_zero} SIREN COMMENÇANT PAR ZÉRO")
print("-" * 50)

# --- VÉRIFICATION 1 : INTÉGRITÉ ET LONGUEUR ---
anomalies_longueur = df_zero[df_zero['SIREN'].apply(len) != 9]
if anomalies_longueur.empty:
    print(f"✅ Longueur : Tous les SIREN '0' font bien 9 caractères.")
else:
    print(f"⚠️ Anomalie : {len(anomalies_longueur)} SIREN n'ont pas la bonne taille.")

# --- VÉRIFICATION 2 : UNICITÉ (DOUBLONS) ---
doublons = df_zero.duplicated(subset=['SIREN']).sum()
if doublons == 0:
    print(f"✅ Unicité : Aucun doublon détecté parmi ces SIREN.")
else:
    print(f"⚠️ Attention : {doublons} SIREN apparaissent plusieurs fois.")

# --- VÉRIFICATION 3 : RÉPARTITION GÉOGRAPHIQUE ---
print(f"\n📍 RÉPARTITION PAR DÉPARTEMENT (Top 10) :")
repartition = df_zero[col_dep].value_counts().head(10)
print(repartition)

# --- VÉRIFICATION 4 : ÉCHANTILLON VISUEL ---
print(f"\n👀 APERÇU DES PREMIÈRES LIGNES :")
cols_view = ['SIREN', 'Dénomination', col_dep, 'Prob_1an']
print(df_zero[cols_view].head())

#### Lancement de l'extraction du sample sur 500 SIREN pour récupération des bilans

In [ ]:
# # 1. Nettoyage et formatage
# df_clean = df.drop_duplicates(subset=['SIREN']).copy()
# df_clean['SIREN'] = df_clean['SIREN'].astype(str).str.zfill(9)

# # 2. Sélection des profils prioritaires (Risque N+1 et N+3)
# top_risques = pd.concat([
#     df_clean.nlargest(200, 'Prob_1an'),
#     df_clean.nlargest(200, 'Prob_3ans')
# ]).drop_duplicates(subset=['SIREN'])

# # 3. Compléter pour atteindre exactement 500
# nb_actuel = len(top_risques)
# nb_a_ajouter = 500 - nb_actuel

# if nb_a_ajouter > 0:
#     reste_df = df_clean[~df_clean['SIREN'].isin(top_risques['SIREN'])]
#     complement = reste_df.sample(n=min(nb_a_ajouter, len(reste_df)))
#     sample_final = pd.concat([top_risques, complement])
# else:
#     sample_final = top_risques.head(500)

# # 4. Export en format LISTE SIMPLE (format JSON array)

# siren_list = sample_final['SIREN'].tolist()

# with open('siren_sample.json', 'w', encoding='utf-8') as f:
#     json.dump(siren_list, f, indent=2)

# print(f"✅ Fichier 'siren_sample.json' généré avec {len(siren_list)} SIREN.")
# print(f"   Structure : ['{siren_list[0]}', '{siren_list[1]}', ...]")

#### Augmentation de l'échantillon à 2000 SIREN

In [ ]:
# # --- CONFIGURATION ---
# TAILLE_CIBLE = 2000
# NB_RISQUE_TOP = 400 

# # 1. Nettoyage
# df_clean = df.drop_duplicates(subset=['SIREN']).copy()
# df_clean['SIREN'] = df_clean['SIREN'].astype(str).str.zfill(9)

# # 2. Sélection (Risque N+1 et N+3)
# top_risques = pd.concat([
#     df_clean.nlargest(NB_RISQUE_TOP, 'Prob_1an'),
#     df_clean.nlargest(NB_RISQUE_TOP, 'Prob_3ans')
# ]).drop_duplicates(subset=['SIREN'])

# # 3. Compléter jusqu'à 2000
# nb_a_ajouter = TAILLE_CIBLE - len(top_risques)
# reste_df = df_clean[~df_clean['SIREN'].isin(top_risques['SIREN'])]
# complement = reste_df.sample(n=min(nb_a_ajouter, len(reste_df)))
# sample_final = pd.concat([top_risques, complement])

# # 4. Export
# siren_list = sample_final['SIREN'].tolist()
# with open('siren_batch_2000.json', 'w', encoding='utf-8') as f:
#     json.dump(siren_list, f, indent=2)

# print(f"✅ Nouveau fichier 'siren_batch_2000.json' prêt !")

---

#### Nouvel echantillonage à 20 000 stuctures

In [ ]:
# # --- CONFIGURATION ---
# TAILLE_CIBLE = 20000  # On passe à 20k pour augmenter massivement ton gisement
# NB_RISQUE_TOP = 5000   # On cible les 5000 plus risqués

# # 1. Chargement des SIREN déjà en base (pour ne pas les reprendre)
# # On utilise le CSV qu'on vient d'extraire du notebook Neon
# df_deja_extraits = pd.read_csv('dataset_bilans_complet.csv')
# sirens_connus = set(df_deja_extraits['siren'].astype(str).str.zfill(9).unique())

# # 2. Nettoyage du dataset Parquet
# df_clean = df.drop_duplicates(subset=['SIREN']).copy()
# df_clean['SIREN'] = df_clean['SIREN'].astype(str).str.zfill(9)

# # 3. FILTRE : On ne garde que les SIREN qu'on n'a PAS encore en base
# df_nouveaux = df_clean[~df_clean['SIREN'].isin(sirens_connus)].copy()
# print(f"🔍 SIREN jamais explorés disponibles : {len(df_nouveaux)}")

# # 4. Sélection des nouveaux (Risque N+1 et N+3)
# top_risques = pd.concat([
#     df_nouveaux.nlargest(NB_RISQUE_TOP, 'Prob_1an'),
#     df_nouveaux.nlargest(NB_RISQUE_TOP, 'Prob_3ans')
# ]).drop_duplicates(subset=['SIREN'])

# # 5. Compléter jusqu'à la TAILLE_CIBLE avec des profils variés
# nb_a_ajouter = TAILLE_CIBLE - len(top_risques)
# reste_df = df_nouveaux[~df_nouveaux['SIREN'].isin(top_risques['SIREN'])]
# complement = reste_df.sample(n=min(nb_a_ajouter, len(reste_df)))

# sample_final = pd.concat([top_risques, complement])

# # 6. Export du nouveau batch
# siren_list = sample_final['SIREN'].tolist()
# with open('siren_batch_20000_nouveaux.json', 'w', encoding='utf-8') as f:
#     json.dump(siren_list, f, indent=2)

# print(f"✅ Nouveau fichier 'siren_batch_20000_nouveaux.json' prêt !")
# print(f"🚀 Prêt à aller chercher les bilans pour {len(siren_list)} nouvelles entreprises.")

#### Vérification du sample

In [ ]:
# import json

# # Ouverture et lecture du fichier
# file_path = 'siren_batch_20000_nouveaux.json'

# with open(file_path, 'r', encoding='utf-8') as f:
#     siren_list_check = json.load(f)

# # Diagnostic rapide
# print(f"📊 Nombre de SIREN dans le fichier : {len(siren_list_check)}")
# print(f"🧐 Type de donnée : {type(siren_list_check)}")
# print(f"🔢 Aperçu des 10 premiers : {siren_list_check[:10]}")

# # Petite vérification de sécurité sur le format
# if all(isinstance(s, str) and len(s) == 9 for s in siren_list_check[:50]):
#     print("✅ Format validé : Tous les SIREN sont des chaînes de 9 caractères.")
# else:
#     print("⚠️ Attention : Certains SIREN n'ont pas le bon format.")
    

---

#### Augmentation à 50 000 SIREN

In [ ]:
# # --- CONFIGURATION DU BATCH #3 ---
# TAILLE_SUPP = 30000 
# NB_RISQUE_TOP = 7500  # On augmente un peu la part de risques élevés

# # 1. Chargement des SIREN déjà en base (historique lointain)
# df_deja_extraits = pd.read_csv('dataset_bilans_complet.csv')
# sirens_connus_bdd = set(df_deja_extraits['siren'].astype(str).str.zfill(9).unique())

# # 2. Chargement du batch de 20k actuellement en cours (pour ne pas les reprendre !)
# with open('siren_batch_20000_nouveaux.json', 'r') as f:
#     batch_20k = set(json.load(f))

# # On fusionne les deux listes d'exclusion
# exclusion_totale = sirens_connus_bdd.union(batch_20k)
# print(f"🚫 Total SIREN à exclure (BDD + Batch actuel) : {len(exclusion_totale)}")

# # 3. Préparation du dataset source (ton gros fichier Parquet 'df')
# df_clean = df.drop_duplicates(subset=['SIREN']).copy()
# df_clean['SIREN'] = df_clean['SIREN'].astype(str).str.zfill(9)

# # 4. FILTRE : On ne garde que les SIREN strictement vierges
# df_nouveaux = df_clean[~df_clean['SIREN'].isin(exclusion_totale)].copy()
# print(f"🔍 SIREN réellement disponibles pour ce nouveau batch : {len(df_nouveaux)}")

# # 5. Sélection par risques (Top Prob_1an et Prob_3ans)
# top_risques = pd.concat([
#     df_nouveaux.nlargest(NB_RISQUE_TOP, 'Prob_1an'),
#     df_nouveaux.nlargest(NB_RISQUE_TOP, 'Prob_3ans')
# ]).drop_duplicates(subset=['SIREN'])

# # 6. Compléter pour atteindre les 30 000
# nb_a_ajouter = TAILLE_SUPP - len(top_risques)
# reste_df = df_nouveaux[~df_nouveaux['SIREN'].isin(top_risques['SIREN'])]

# if nb_a_ajouter > 0:
#     complement = reste_df.sample(n=min(nb_a_ajouter, len(reste_df)))
#     sample_final = pd.concat([top_risques, complement])
# else:
#     sample_final = top_risques

# # 7. Export du nouveau batch spécifique
# siren_list_30k = sample_final['SIREN'].tolist()
# filename = 'siren_batch_30000_supplementaires.json'

# with open(filename, 'w', encoding='utf-8') as f:
#     json.dump(siren_list_30k, f, indent=2)

# print("-" * 50)
# print(f"✅ Fichier '{filename}' généré !")
# print(f"🚀 Nouveau gisement : {len(siren_list_30k)} entreprises uniques prêtes pour l'INPI.")

---

#### Ajout de 100 000 nouveaux SIREN dans l'échantillon

In [ ]:
import pandas as pd
import json
import os

# --- CONFIGURATION DU BATCH #4 ---
TAILLE_SUPP = 100000 
NB_RISQUE_TOP = 20000  # On cible un gros volume de risques pour ce gros batch

# 1. Chargement de l'historique lointain (BDD)
df_deja_extraits = pd.read_csv('dataset_bilans_complet.csv')
sirens_connus_bdd = set(df_deja_extraits['siren'].astype(str).str.zfill(9).unique())

# 2. Chargement des batches intermédiaires (Batch 2 et Batch 3)
# On ratisse large pour ne rien reprendre par erreur
batches_precedents = []
for f_name in ['siren_batch_20000_nouveaux.json', 'siren_batch_30000_supplementaires.json']:
    if os.path.exists(f_name):
        with open(f_name, 'r') as f:
            batches_precedents.extend(json.load(f))

sirens_batches = set(str(s).zfill(9) for s in batches_precedents)

# Fusion de toutes les exclusions
exclusion_totale = sirens_connus_bdd.union(sirens_batches)
print(f"🚫 Total SIREN à exclure (BDD + Batch 2 + Batch 3) : {len(exclusion_totale)}")

# 3. Préparation du dataset source
df_clean = df.drop_duplicates(subset=['SIREN']).copy()
df_clean['SIREN'] = df_clean['SIREN'].astype(str).str.zfill(9)

# 4. FILTRE : On ne garde que les SIREN vierges
df_nouveaux = df_clean[~df_clean['SIREN'].isin(exclusion_totale)].copy()
print(f"🔍 SIREN réellement disponibles : {len(df_nouveaux)}")

# 5. Sélection par risques
top_risques = pd.concat([
    df_nouveaux.nlargest(NB_RISQUE_TOP, 'Prob_1an'),
    df_nouveaux.nlargest(NB_RISQUE_TOP, 'Prob_3ans')
]).drop_duplicates(subset=['SIREN'])

# 6. Compléter pour atteindre les 100 000
nb_a_ajouter = TAILLE_SUPP - len(top_risques)
reste_df = df_nouveaux[~df_nouveaux['SIREN'].isin(top_risques['SIREN'])]

if nb_a_ajouter > 0:
    complement = reste_df.sample(n=min(nb_a_ajouter, len(reste_df)))
    sample_final = pd.concat([top_risques, complement])
else:
    sample_final = top_risques

# 7. Export du Batch #4
siren_list_100k = sample_final['SIREN'].tolist()
filename = 'siren_batch_100000_supplementaires.json'

with open(filename, 'w', encoding='utf-8') as f:
    json.dump(siren_list_100k, f, indent=2)

print("-" * 50)
print(f"✅ Fichier '{filename}' généré !")
print(f"🚀 Nouveau gisement : {len(siren_list_100k)} entreprises uniques.")